# 02 · Section extraction — the step that fails silently

Pull **Item 1A (Risk Factors)** and **Item 7 (MD&A)** out of every filing.

This is the highest-risk component in the project, and the reason is worth
stating precisely, because "I used a regex" is not a defensible answer to
anyone who knows filings.

Every 10-K contains the string *"Item 1A. Risk Factors"* **at least twice** —
once in the table of contents near the front, once at the real section — plus
cross-references like *"see Item 1A. Risk Factors below"* elsewhere in the
document. A regex that takes the first match extracts **the table of
contents** and the couple of lines after it.

That output is not empty and not obviously wrong. It is a short string of
plausible-looking text. Nothing downstream raises an error, the pipeline runs
green, and every number it produces is meaningless.

**The rule used here:** find every candidate anchor for the opening item and
every anchor for the following item, form all valid `(start, end)` spans, and
keep the **longest**. The real section is typically an order of magnitude
longer than a contents line, so length discriminates far more reliably than
position.

Each span is then length-audited, and two failure modes are kept **separate**:

| status | meaning |
|---|---|
| `ok` | usable section, bounded by a real end anchor |
| `absent` | the company legitimately omitted Item 1A — smaller reporting companies may |
| `tooshort` | below the minimum plausible length |
| `noanchor` | no Item 1A heading anywhere |
| `ok_loose` | usable, but located only by the fallback anchor tier |
| `unbounded` | found a start but no end anchor after it |
| `runon` | span longer than any real section — a missed end anchor |

`ok` and `ok_loose` are both usable. They are counted separately because
`ok_loose` rests on weaker evidence — the heading matched mid-line rather
than at the start of one — and folding it into `ok` would launder that
distinction away. The rest are kept apart for the same reason: pooling "the
company omitted this" with "we failed to find it" would put parsing noise
into the year-over-year comparison the whole project rests on.

### Why there are two anchor tiers

Requiring headings to begin a line is the right default — it excludes
mid-sentence mentions outright. But filings put headings inside **layout
tables**, and a heading sitting in the second cell of a row lands mid-line
in the extracted text. Enforcing a line start alone reported 2,571 filings
(29.6%) as `noanchor` and cut the year-over-year sample from 5,151 to 3,760.

So strict anchors run first, and only when they yield nothing usable does
the loose tier retry with position-independent anchors. The length floor,
the longest-span rule and the run-on guard apply on both passes, which is
what stops a mid-sentence match from producing the kind of fragment
described below.

### The bug this design exists to prevent

An earlier version of this extractor gave a start anchor with no end after
it a fixed **200,000-character window**, so that a final section would still
be recoverable. Combined with *longest span wins*, that was a trap.

A stray late reference — `"see Item 1A. Risk Factors"` in an exhibit index or
a signature block, sitting after the real Item 1B — has no end anchor
following it. It inherited the full 200,000-character window and **beat every
genuine span**. On the real panel that produced ~3,800 filings with a section
length of exactly 200,000 characters: a constant masquerading as a
measurement, sitting inside a column that looked entirely reasonable in
aggregate.

The fix: spans with a real end anchor are the only ones that compete.
Unanchored starts are reported as `unbounded`, never as `ok`, so they cannot
silently enter the analysis.

**The diagnostic that caught it is below** — a check for the most common
*exact* length. If many filings share one precise length, that length is
coming from the code, not from the companies.

In [ ]:
import sys, warnings, os
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

import config, viz
from sections import extract, extract_all, MIN_SECTION_CHARS
viz.use_style()
print(config.describe())

## The extractor, on one filing

Before running 8,800, look at one. This is the habit that catches the
table-of-contents bug.

In [ ]:
from sec_client import read_gz

sample = sorted(config.FILINGS.glob("*.txt.gz"))[0]
text = read_gz(sample)
print(f"{sample.name}   {len(text):,} chars\n")

secs = extract_all(text)
for name, s in secs.items():
    print(f"{name:<8} status={s.status:<9} chars={s.n_chars:>8,} "
          f"anchors_seen={s.n_candidates}")

`anchors_seen` above 1 is the normal case, and it is the whole point — the
extractor saw several candidates and chose between them. Now read the first
400 characters of what it chose. It should be **prose about risks**, not a
list of page numbers.

In [ ]:
print(secs["item1a"].text[:400])
print("\n" + "-"*70 + "\n")
print(secs["item7"].text[:400])

## Run over the whole panel

~3–5 minutes across 4 processes.

> **Kernel note.** `import x` is a no-op if `x` was already imported in
this kernel, so an edited `.py` file silently does not load and the cell
happily reports numbers from the previous version. Every cell below that
calls into a pipeline module reloads it first, in dependency order.

In [ ]:
import importlib
import sections, extract_sections
importlib.reload(sections)          # reload dependencies first
importlib.reload(extract_sections)

extract_sections.main(workers=4)

## Diagnostics

The status mix is the first thing to read. Rough expectations on a panel of
listed filers: `ok` should dominate; a few percent `absent` is normal
(smaller reporting companies); **`tooshort` above ~15% means the anchors need
widening** and nothing built on top of this would be trustworthy.

In [ ]:
df = pd.read_parquet(config.INTERIM / "sections.parquet")
n = len(df)

status = (df["item1a_status"].value_counts().rename("filings").to_frame()
          .assign(share=lambda d: (d["filings"]/n).map("{:.1%}".format)))
display(status)

usable = df["item1a_status"].isin(["ok", "ok_loose"])
print(f"usable Item 1A: {usable.sum():,} of {n:,}  ({usable.mean():.1%})")

### Length distribution

A real risk-factors section runs tens of thousands of characters. A spike
down at the `MIN_SECTION_CHARS` floor would mean the extractor is scraping
contents lines and barely clearing the threshold.

In [ ]:
s = df.loc[usable, "item1a_chars"]

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist(s, bins=60, color=viz.SERIES[0], edgecolor=viz.SURFACE, linewidth=0.5)
ax.axvline(MIN_SECTION_CHARS, color=viz.SERIES[7], linewidth=2)
ax.annotate(f"  floor {MIN_SECTION_CHARS:,}", (MIN_SECTION_CHARS, ax.get_ylim()[1]*0.9),
            color=viz.SERIES[7], fontsize=9, fontweight="600")
ax.set_title("Item 1A length")
ax.set_xlabel("characters"); ax.set_ylabel("filings")
plt.show()

print(s.describe(percentiles=[.01,.10,.50,.90,.99]).round(0).to_string())

### The constant-length check

**The single most useful diagnostic here.** Real companies do not write
sections of identical length. If many filings share one *exact* character
count, that number is coming from the code — a default, a window, a
truncation — and not from the data.

This is the check that caught the 200,000-character window described above.
It looked fine in every aggregate: the mean was plausible, the distribution
had a sensible left tail. Only the exact-value count revealed that half the
column was one repeated number.

In [ ]:
ok_len = df.loc[usable, "item1a_chars"]
top = ok_len.value_counts().head(5).rename("filings").to_frame()
top["share"] = (top["filings"] / len(ok_len)).map("{:.2%}".format)
display(top)

worst = top["filings"].iloc[0] / len(ok_len)
print("SUSPICIOUS — one exact length dominates; find where it comes from"
      if worst > 0.02 else
      f"ok — most common exact length is only {worst:.2%} of filings")

### Run-on sections

If the end anchor (Item 1B) is missed, a section runs on into the rest of the
filing. The extractor now rejects these outright as `runon`, but the share of
the document is still worth eyeballing for the ones that passed.

In [ ]:
frac = (df["item1a_chars"] / df["doc_chars"].clip(lower=1)).where(usable)
runaway = df[usable & (frac > 0.60)]
print(f"Item 1A > 60% of the filing: {len(runaway):,}  "
      f"({len(runaway)/usable.sum():.2%})")
if len(runaway):
    display(runaway[["ticker","filing_year","item1a_chars","doc_chars"]].head(10))

### Read the extremes by hand

Two minutes here is worth more than any automated check. Look at the shortest
usable section and the longest — the shortest is where a contents-line bug
hides, the longest is where a missed end-anchor hides.

In [ ]:
# Join on the manifest's `local` column rather than rebuilding the
# filename — the accession number is part of it and reconstructing it by
# hand is a good way to silently look at the wrong filing.
man = pd.read_parquet(config.MANIFEST)[["cik","filing_year","local"]]
d = df[usable].merge(man, on=["cik","filing_year"], how="left")

for tag, row in [("SHORTEST", d.nsmallest(1,"item1a_chars").iloc[0]),
                 ("LONGEST",  d.nlargest(1,"item1a_chars").iloc[0])]:
    sec = extract(read_gz(config.FILINGS / row["local"]), "item1a")
    print(f"=== {tag}  {row['ticker']} {row['filing_year']}  "
          f"{sec.n_chars:,} chars ===")
    print(sec.text[:350].replace("\n", " "), "\n")

### The panel that survives

The year-over-year feature needs at least two consecutive usable years per
firm. This count — not the raw filing count — is the real sample size for the
headline result.

In [ ]:
per_firm = df[usable].groupby("cik")["filing_year"].apply(sorted)

def consecutive_pairs(years):
    return sum(1 for a, b in zip(years, years[1:]) if b - a == 1)

pairs = per_firm.map(consecutive_pairs)
print(f"firms with >=1 consecutive usable pair : {(pairs>0).sum():,}")
print(f"total year-over-year observations      : {pairs.sum():,}")
print(f"  <- this is the sample size for risk_change")